# 02 · Exploratory Spatial Analysis
**Chicago Energy Burden project, Stage 2 · Research Question 1: Where is energy burden concentrated?**

This notebook answers two questions:
1. **Is burden clustered across the city as a whole?** (Global Moran's I)
2. **Where exactly are the hotspots?** (Local Moran's I, also called LISA)

**To run it:** choose **Runtime > Run all**. If Stage 1 hasn't been run in this session, the notebook runs it first.

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("Chicago-Energy-Burden").exists():
        !git clone -q https://github.com/dayyan24-netizen/Chicago-Energy-Burden.git
    os.chdir("Chicago-Energy-Burden")
    !pip install -q -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, "src")
from ceb import config, download, build, explore

if not (config.PROCESSED / "chicago_tracts.gpkg").exists():
    print("Stage 1 outputs not found, running Stage 1 first...")
    lead_zip = download.download_lead()
    tract_zip, place_zip = download.download_tiger()
    cov, cost = download.download_acs()
    build.run(tract_zip, place_zip, lead_zip, cov, cost)
print("Ready.")

## How to read Moran's I
Moran's I asks: **do tracts look like their neighbors?**

* **I near 0:** burden is scattered randomly across the map.
* **I above 0:** high-burden tracts sit next to other high-burden tracts (clustering).
* **I below 0:** high and low tracts alternate like a checkerboard (rare in real cities).

To test significance, the computer shuffles the burden values randomly across tracts 9,999 times and checks how often a random map is as clustered as the real one. That share is the *permutation p-value*.

LISA runs the same idea for **each tract**: is this tract, together with its neighbors, unusually high or low? Because we run one test per tract, some would look significant by chance alone, so we use a **false discovery rate (FDR)** correction to keep false hotspots to about 5% of those flagged.

In [ ]:
community_areas = download.download_community_areas()
gdf = explore.run(community_areas)

## Results report

In [ ]:
from IPython.display import Markdown, display
report = (config.REPORTS / "spatial_analysis_report.md").read_text()
display(Markdown("\n".join(l for l in report.splitlines() if not l.startswith("!["))))

In [ ]:
from IPython.display import Image
for f in ["small_multiples.png", "moran_scatter_burden.png", "lisa_burden.png", "lisa_share_severe.png"]:
    display(Image(str(config.REPORTS / "figures" / f), width=520))

## Interactive cluster map
Hover over a tract to see its community area, burden, and cluster type.

In [ ]:
colors = explore.CLUSTER_COLORS
gdf.explore(
    column="burden_cluster_fdr",
    categories=list(colors), cmap=list(colors.values()),
    tooltip=["GEOID", "community_area", "burden", "median_hh_income", "burden_cluster_fdr"],
    legend_kwds={"caption": "LISA cluster (FDR 5%)"},
    tiles="CartoDB positron")

## Hotspot tracts
Every tract in a statistically significant high-burden cluster, highest burden first.

In [ ]:
cols = ["GEOID", "community_area", "burden", "share_severe", "median_hh_income",
        "pct_black", "pct_hispanic", "pct_pre1950", "pct_renter"]
hot = gdf[gdf["burden_cluster_fdr"] == "High-High"][cols].sort_values("burden", ascending=False)
print(f"{len(hot)} hotspot tracts")
hot.round(1).reset_index(drop=True)

## Questions to bring to your advisors
1. Is Moran's I similar under queen and k-nearest-neighbor weights? If yes, the clustering is not an artifact of tract shapes.
2. Which community areas hold the hotspots, and do they match ACEEE and Elevate's descriptions of the South and West Sides?
3. How many tracts are significant at p < 0.05 but drop out under FDR? Those are the borderline cases to mention in the paper.
4. Do the hotspots differ from other tracts mainly in income, or also in housing age and renting? Stage 3 tests this formally.